# Qulture.Rocks API

Templates de uso para coletar dados da QR
- Documentação: https://docs.qulture.rocks/docs/how-to-use-qulturerocks-api

### Qulture.rocks integration API (export personalizado)
- documentação: https://app.qulture.rocks/apidoc
- endpoint: https://app.qulture.rocks/api_integration/
- token: <token1>


### REST API
Substituir XXXX pelo id da empresa e YYYYY pelo id da pesquisa
- documentação: https://docs.qulture.rocks/reference
- endpoint: "https://api.qulture.rocks/rest/companies/XXXX/
- token: <token2

## Listar contracts (usuários)

In [ ]:
import requests
import pandas as pd

# Preenchendo o token
token = "<token1>"
# Criando o tipo de acesso
headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {token}"
}

# Inicializando variáveis
url_base = "https://app.qulture.rocks/api_integration/contracts"
page = 1
seen_ids = set()  # Para armazenar IDs já vistos e evitar duplicatas
all_data = []  # Para armazenar todos os dados coletados

while True:
    url_members = f"{url_base}?page={page}"  # Atualiza a URL com a nova página
    response = requests.get(url=url_members, headers=headers)
    
    # Interrompe o loop se a chamada receber um erro
    if response.status_code != 200:
        break
    
    response_members = response.json()
    dados = pd.Series(response_members).to_frame(name='dados')
    dados = dados.explode(column='dados', ignore_index=True)
    
    # Se não houver dados, interrompe o loop
    if dados.empty:
        break
    
    df = pd.json_normalize(data=dados['dados'])
    
    # Se todos os IDs nesta página já foram vistos, interrompe o loop
    current_ids = set(df['id'].unique())
    if current_ids.issubset(seen_ids):
        break
    else:
        seen_ids.update(current_ids)  # Atualiza o conjunto de IDs vistos
    
    all_data.append(df)  # Adiciona os dados da página atual à lista de todos os dados
    page += 1  # Prepara para a próxima página

# Concatena todos os DataFrames na lista all_data para criar um único DataFrame
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    #print(final_df)
else:
    print("Nenhum dado foi coletado.")


contratos = final_df#[['id','name','area','department','email','job_title']]

# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/contratos.xlsx"
contratos.to_excel(excel_filename, index=False)

#print(f"Arquivo Excel salvo como: {excel_filename}")

contratos

## Coletar notas (Team Results)

In [ ]:
# Substituir XXXX pelo id da empresa e YYYYY pelo id da pesquisa

import requests
from pandas import json_normalize
import pandas as pd

base_url = "https://api.qulture.rocks/rest/companies/XXXX/surveys/YYYYY/survey_participations?include="
headers = {
    "accept": "application/json",
    "Authorization": "Bearer <token2>"
}

df_total = pd.DataFrame()  # Dataframe para armazenar todos os dados
page = 1
last_df = None  # Para armazenar a resposta da última página para comparação

while True:
    current_url = f"{base_url}&page={page}"
    response = requests.get(current_url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        
        if 'survey_participations' in data:
            df_page = json_normalize(data['survey_participations'])
            
            if df_page.empty or df_page.equals(last_df):
                # Se o DataFrame da página atual está vazio ou é igual ao último, termina o loop
                break
            
            df_total = pd.concat([df_total, df_page], ignore_index=True)
            last_df = df_page  # Atualiza o último DataFrame para comparação na próxima iteração
            
            page += 1
        else:
            break
    else:
        print(f"Erro na requisição: {response.status_code}")
        break

# Exibe o DataFrame total após coletar todos os dados
#print(df_total)

notas = df_total
'''[['id','contract_id','survey_id','grade_final','reviews_not_started_count','reviews_started_count',
    'reviews_completed_count','reviews_submitted_count','reviews_count','review_progress_percentage',
    'grades_averages.others','grades_averages.average','grades_averages.colleagues','grades_averages.from_peers',
    'grades_averages.self_review','grades_averages.no_self_review','grades_averages.from_supervisor',
    'grades_averages.from_subordinates','grades_averages.from_subordinates_peers_and_others']]'''


# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/notas.xlsx"
notas.to_excel(excel_filename, index=False)

notas

### Unir bases

In [ ]:
import pandas as pd
import numpy as np

# Criando DataFrames
A = contratos
B = notas

# Realizando o left join
base_avaliacao_completa = pd.merge(A, B, left_on='id', right_on='contract_id', how='left')

# Criar colunas novas
base_avaliacao_selecionada = base_avaliacao_completa
base_avaliacao_selecionada['Nota Final (sem calibração)'] = base_avaliacao_selecionada['grades_averages.from_supervisor']
base_avaliacao_selecionada['Identificador'] = np.nan
base_avaliacao_selecionada['Status'] = base_avaliacao_selecionada['active'].apply(lambda x: 'Ativo' if x else 'Inativo')
base_avaliacao_selecionada['Times durante a avaliação'] = np.nan
base_avaliacao_selecionada['Tags durante a avaliação'] = np.nan
base_avaliacao_selecionada['Relatório individual visto em'] = np.nan
base_avaliacao_selecionada['Resultados liberados'] = np.nan
base_avaliacao_selecionada['Data de Agendamento de devolutiva'] = np.nan 
base_avaliacao_selecionada['Devolutiva concluida?'] = np.nan 
base_avaliacao_selecionada['Data de Conclusão da devolutiva'] = np.nan 

# Mapeando 'name' para 'supervisor_id' usando 'id_x' como chave
# Cria um dicionário onde 'id_x' é a chave e 'name' é o valor
name_map = dict(zip(base_avaliacao_selecionada['id_x'], base_avaliacao_selecionada['name']))
# Usando o map para aplicar o dicionário e criar a nova coluna 'Líder'
base_avaliacao_selecionada['Líder'] = base_avaliacao_selecionada['supervisor_id'].map(name_map)


# Função para converter valores numéricos em rótulos de texto
def valor_para_rotulo(valor):
    if pd.isnull(valor):  # pd.isnull() captura tanto np.nan quanto None
        return ''  # Retorna uma string vazia para valores vazios
    # Depois, verifica os intervalos
    elif valor == 0:
        return ''
    elif 0 < valor < 1.0:
        return 'q4'
    elif 1.0 <= valor < 2.0:
        return 'q2'
    elif 2.0 <= valor < 3.0:
        return 'q3'
    elif 3.0 <= valor <= 4.0:
        return 'q4'
    else:
        return 'Fora de Faixa'  # Para valores fora dos intervalos especificados

# Lista de colunas a serem modificadas
colunas = [
    'grade_final', 'grades_averages.average','grades_averages.self_review','grades_averages.from_supervisor',
    'grades_averages.others','grades_averages.colleagues','grades_averages.from_peers','grades_averages.no_self_review',
    'grades_averages.from_subordinates','Nota Final (sem calibração)','calibration'
]

# Aplicando a conversão para cada coluna especificada
for coluna in colunas:
    base_avaliacao_selecionada[coluna] = base_avaliacao_selecionada[coluna].apply(valor_para_rotulo)



# Selecionando colunas
base_avaliacao_selecionada = base_avaliacao_completa[['name','email','Identificador','cpf','area','department','Status',
                                                      'Times durante a avaliação','Tags durante a avaliação','Líder',
                                                      'location','Relatório individual visto em','Resultados liberados',
                                                      'grade_final','Nota Final (sem calibração)',
                                                      'calibration','has_calibration','grades_averages.average',
                                                      'grades_averages.self_review','grades_averages.from_supervisor',
                                                      'grades_averages.others','grades_averages.colleagues',
                                                      'grades_averages.from_peers','grades_averages.no_self_review',
                                                      'grades_averages.from_subordinates','Data de Agendamento de devolutiva',
                                                      'Devolutiva concluida?','Data de Conclusão da devolutiva'
                                                      ]]


# Dicionário para mapear nomes antigos de colunas para novos nomes
rename_dict = {
    "name": "Nome",
    "email": "Email",
    "cpf": "CPF",
    "area": "Área durante a avaliação",
    "department": "Departamento durante a avaliação",
    "location": "Localização",
    "grade_final": "Nota Final (com calibração)",
    "calibration": "Nota calibração",
    "has_calibration": "Houve calibração?",
    "grades_averages.average": "Média",
    "grades_averages.self_review": "Autoavaliação",
    "grades_averages.from_supervisor": "Do Líder",
    "grades_averages.others": "Outros",
    "grades_averages.colleagues": "Colegas",
    "grades_averages.from_peers":"Dos Pares",
    "grades_averages.no_self_review": "Média Sem Autoavaliação",
    "grades_averages.from_subordinates": "Dos Subordinados"
    }


# Renomeando as colunas do DataFrame
base_avaliacao_renomeada = base_avaliacao_selecionada.rename(columns=rename_dict)


base_avaliacao = base_avaliacao_renomeada

# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/base_avaliacao.xlsx"
base_avaliacao.to_excel(excel_filename, index=False)

# Exibindo o resultado
#base_avaliacao
base_avaliacao

## Interagindo com a planilha do Google sheets


In [ ]:
#pip install google-api-python-client
import requests
import numpy as np
import json
import pandas as pd
import datetime as dt
import os
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from __future__ import print_function

# Caso receba erro SSL, rodar a seguinte linha
#!pip install --upgrade requests urllib3 pyOpenSSL

### Lê a base atual

In [ ]:
# Código se vc tem credentials e token em arquivos json 
'''
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
## criando o acesso
creds = None
if os.path.exists('token.json'):
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            'credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
try:
## chamando a planilha
    service = build('sheets', 'v4', credentials=creds)
    sheet = service.spreadsheets()
    df_base_antiga = sheet.values().get(spreadsheetId='XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX', 
                                       range='Sheet1!A1:Z').execute()
    df_base_antiga = df_pedidos_hist.get("values",[])
except:
    print(0)
'''

# Código se quer imbutir credentials e json dentro do código (ideal para conversão em executável)
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

# Dados do credentials.json como uma string
credentials_json = """
<conteúdo do json>
"""

# Dados do token.json como uma string
# Assumindo que você tenha um token de refresh válido
token_json = """
<conteúdo do json>
"""

# Criando o acesso
creds = None
if token_json:
    creds = Credentials.from_authorized_user_info(json.loads(token_json), SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_config(json.loads(credentials_json), SCOPES)
        creds = flow.run_local_server(port=0)
        # Simulação de salvar o token (em memória, não em arquivo)
        token_json = creds.to_json()  # Atualiza o token_json com o novo token

# Tentativa de acessar a planilha do Google Sheets
try:
    service = build('sheets', 'v4', credentials=creds)
    sheet = service.spreadsheets()
    df_base_antiga = sheet.values().get(
        spreadsheetId='XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX',
        range='Sheet1!A1:Z').execute()
    df_base_antiga = df_base_antiga.get("values", [])
    #print(df_base_antiga)  # Exibir os dados recuperados
except Exception as e:
    print("Erro ao acessar a planilha: ", e)
    print(0)


## Criando o data frame
df_base_antiga = pd.DataFrame(df_base_antiga, columns = [
    'Nome','Email','Identificador','CPF','Área durante a avaliação', 'Departamento durante a avaliação','Status',
    'Times durante a avaliação','Tags durante a avaliação','Líder','Localização','Relatório individual visto em',
    'Resultados liberados','Nota Final (com calibração)','Nota Final (sem calibração)','Média','Autoavaliação',
    'Do Líder','Outros','Colegas','Dos Pares','Média Sem Autoavaliação','Dos Subordinados',
    'Data de Agendamento de devolutiva','Devolutiva concluida?','Data de Conclusão da devolutiva'])
df_base_antiga

### Apaga o conteúdo do arquivo atual e substituiu pelo novo

In [ ]:
# Enviando os dados de volta para a planilha
# caso receba erro de SSL, rodar:
#pip install --upgrade requests urllib3 pyOpenSSL

df = base_avaliacao

# Converte o DataFrame para uma lista de listas
df.fillna(value=0, inplace=True)
data = df.values.tolist()

try:
    result = sheet.values().update(
        spreadsheetId='XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX',
        range='Sheet1!A2',  # Define o canto superior esquerdo como o ponto de partida
        valueInputOption='RAW',
        #body={'values': df}
        body={'values': data}        
    ).execute()
    print('Quantidade de células atualizadas: {0}'.format(result.get('updatedCells')))
except HttpError as e:
    print(f"Erro ao enviar dados de volta para a planilha: {e}")

## Agendamento automático

1) Comentar todas as etapas de exportação de arquivos (possível problema com endereço de rede das pastas)
Caso vá exportar arquivos, garantir que está usando o nome de caminho completo de pastas

2) Converter o notebook em script: !jupyter nbconvert --to script nome_notebook.ipynb

3) Converter o arquivo .py gerado para executável com o pyinstaller:
pip install pyinstaller
python -m PyInstaller --onefile API_QR_NOTAS.py

4) Na pasta do script deve aparecer um arquivo .spec (detalhes do projeto) e uma pasta chamada 'dist' com o executável

In [ ]:
#!jupyter nbconvert --to script API_QR_NOTAS.ipynb

## Puxando avaliações individuais
1) Baixa toda a base de indicações
2) Para cada indicação, baixa as respostas para cada pergunta

In [ ]:
# Substituir XXXX pelo id da empresa e YYYYY pelo id da pesquisa

import requests
from pandas import json_normalize
import pandas as pd

base_url = "https://api.qulture.rocks/rest/companies/XXXX/surveys/YYYYY/participants"
headers = {
    "accept": "application/json",
    "Authorization": "Bearer <token2>"
}

df_total = pd.DataFrame()  # Dataframe para armazenar todos os dados
page = 1
last_df = None  # Para armazenar a resposta da última página para comparação


while True:
    current_url = f"{base_url}?page={page}"
    response = requests.get(current_url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        
        if 'participants' in data:
            df_page = json_normalize(data['participants'])
            
            if df_page.empty or df_page.equals(last_df):
                # Se o DataFrame da página atual está vazio ou é igual ao último, termina o loop
                break
            
            df_total = pd.concat([df_total, df_page], ignore_index=True)
            last_df = df_page  # Atualiza o último DataFrame para comparação na próxima iteração
            
            page += 1
        else:
            break
    else:
        print(f"Erro na requisição: {response.status_code}")
        break

# Exibe o DataFrame total após coletar todos os dados
#print(df_total)

participants = df_total


# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/participants.xlsx"
participants.to_excel(excel_filename, index=False)

participants

### Une a base com a identificação das pessoas

In [ ]:
import pandas as pd
import numpy as np

# Pega base com dados pessoais por ID


# Criando DataFrames
A = participants
B = contratos[['id','name','job_title','level','area','department']]

# Realizando o left join 
base_avaliacao_1 = pd.merge(A, B, left_on='reviewer_id', right_on='id', how='left')
base_avaliacao_2 = pd.merge(base_avaliacao_1, B, left_on='reviewee_id', right_on='id', how='left')


base_avals_ind = base_avaliacao_2[['completed','is_complete','finished','survey_participation_id','state','kind',
                                   'anonymized?','relationship',
                                   'id_x','name_x','job_title_x','level_x','area_x','department_x',
                                   'id_y','name_y','job_title_y','level_y','area_y','department_y']]
                                   
                                   
# Dicionário para mapear nomes antigos de colunas para novos nomes
rename_dict = {
    "id_x": "id_avaliador",
    "name_x": "nome_avaliador",
    "job_title_x": "cargo_avaliador",
    "level_x": "nivel_avaliador",
    "area_x": "area_avaliador",
    "department_x": "departamento_avaliador",
    "id_y": "id_avaliado",
    "name_y": "nome_avaliado",
    "job_title_y": "cargo_avaliado",
    "level_y": "nivel_avaliado",
    "area_y": "area_avaliado",
    "department_y": "departamento_avaliado"
}
                                   
 
# Renomeando as colunas do DataFrame
base_avals_ind = base_avals_ind.rename(columns=rename_dict)

                                   
# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/base_avaliacoes_individuais.xlsx"
base_avals_ind.to_excel(excel_filename, index=False)

# Exibindo o resultado
base_avals_ind = pd.DataFrame(base_avals_ind)
base_avals_ind

### Baixa respostas individuais de cada pergunta para cada avaliação

In [ ]:
# Substituir XXXX pelo id da empresa e YYYYY pelo id da pesquisa

import requests
from pandas import json_normalize
import pandas as pd

# Configurações da API
base_url = "https://api.qulture.rocks/rest/companies/XXXX/survey_participations/"
headers = {
    "accept": "application/json",
    "Authorization": "Bearer <token2>"
}


df = base_avals_ind.loc[base_avals_ind['nome_avaliado'].isin(['Jose Oliveira', 
                                                              'Maria Santos', 
                                                              'João da Silva'])]


# Inicializa o DataFrame para armazenar todos os dados
df_total = pd.DataFrame()

# Suponha que o DataFrame `df` já está definido e contém a coluna 'survey_participation_id'
for participation_id in df['survey_participation_id']:
    page = 1
    last_df = None  # Para armazenar a resposta da última página para comparação
    while True:
        current_url = f"{base_url}{participation_id}/answers?include=participant&include=Creviewe&page={page}"
        response = requests.get(current_url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            
            if 'answers' in data:
                df_page = json_normalize(data['answers'])
                
                # Adicionando o 'survey_participation_id' como uma coluna em 'df_page'
                df_page['survey_participation_id'] = participation_id

                if df_page.empty or df_page.equals(last_df):
                    # Se o DataFrame da página atual está vazio ou é igual ao último, termina o loop
                    break
                
                df_total = pd.concat([df_total, df_page], ignore_index=True)
                last_df = df_page  # Atualiza o último DataFrame para comparação na próxima iteração
                
                page += 1
            else:
                break
        else:
            print(f"Erro na requisição: {response.status_code}")
            break

# Exibe o DataFrame total após coletar todos os dados
#print(df_total)

respostas = df_total

# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/respostas.xlsx"
respostas.to_excel(excel_filename, index=False)

respostas

In [1]:
Parar execução com erro aqui

SyntaxError: invalid syntax (1393322483.py, line 1)

## ---- Desenvolvimentos futuros ----

Até aqui está tudo certo e funcionando. 
A próxima etapa seria buscar a base de avaliações individuais, i.e. cada pergunta/resposta feita por cada avaliador para cada avaliado, mas esbarrei numa limitação de chaves. 
Não consegui identificar algum id que seja único pra uma 'participant' (par avaliador / avaliado) e que esteja presente tanto na base de 'participants' quanto na base de respostas individuais.

### Lista e respostas perguntas por ID de participant

In [ ]:
# Substituir XXXX pelo id da empresa e YYYYY pelo id da pesquisa

import pandas as pd
import numpy as np

# Pega base de perguntas por ID
import requests
url = "https://api.qulture.rocks/rest/companies/XXXX/surveys/YYYYY/topics?include=questions,question_topic&page=1"
headers = {
    "accept": "application/json",
    "Authorization": "Bearer <token2>"
}
response = requests.get(url, headers=headers)
data = response.json()
dic_perguntas = json_normalize(data, record_path=['topics', 'questions'], meta=[['topics', 'id'], ['topics', 'name']], errors='ignore')
dic_perguntas[['id','name','topics.id','topics.name']]


# Criando DataFrames
A = respostas
B = dic_perguntas

# Realizando o left join 
base_resps_ind = pd.merge(A, B, left_on='question_id', right_on='id', how='left')


base_resps_ind = base_resps_ind[['survey_participation_id','topic.name','name','grading','grade','comment']]

                                
# Dicionário para mapear nomes antigos de colunas para novos nomes
rename_dict = {
    "topic.name": "Tema",
    "name": "pergunta",
    "grading": "nota_num",
    "grade": "nota_cat",
    "comment": "comentario"
}
                                   
 
# Renomeando as colunas do DataFrame
base_resps_ind = base_resps_ind.rename(columns=rename_dict)
                                   
# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/base_respostas_individuais.xlsx"
base_resps_ind.to_excel(excel_filename, index=False)

# Exibindo o resultado
base_resps_ind = pd.DataFrame(base_resps_ind)

base_resps_ind

### Adiciona cada pergunta e resposta à base de participants

In [ ]:
import pandas as pd
import numpy as np

# Pega base com dados pessoais por ID


# Criando DataFrames
A = base_resps_ind
B = respostas

# Realizando o left join 
base_pergs_resps = pd.merge(A, B, left_on='survey_participation_id', right_on='survey_participation_id', how='left')
 

base_pergs_resps = base_pergs_resps
'''
[['completed','is_complete','finished','survey_participation_id','state','kind',
                                   'anonymized?','relationship',
                                   'id_x','name_x','job_title_x','level_x','area_x','department_x',
                                   'id_y','name_y','job_title_y','level_y','area_y','department_y']]
'''
# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/base_pergs_resps.xlsx"
base_pergs_resps.to_excel(excel_filename, index=False)

# Exibindo o resultado
base_pergs_resps = pd.DataFrame(base_pergs_resps)

base_pergs_resps

In [ ]:
pd.merge(A, B, left_on='survey_participation_id', right_on='survey_participation_id', how='left')

In [ ]:
# Substituir XXXX pelo id da empresa e YYYYY pelo id da pesquisa

import requests

url = "https://api.qulture.rocks/rest/companies/XXXX/surveys/YYYYY/topics?include=questions,question_topic&page=1"

headers = {
    "accept": "application/json",
    "Authorization": "Bearer <token2>"
}

response = requests.get(url, headers=headers)

data = response.json()

dic_perguntas = json_normalize(data, record_path=['topics', 'questions'], meta=[['topics', 'id'], ['topics', 'name']], errors='ignore')
dic_perguntas[['id','name','topics.id','topics.name']]


# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/teste.xlsx"
dic_perguntas.to_excel(excel_filename, index=False)

dic_perguntas

### Une a base de 'participants' (par avaliador-avaliado) com as respostas

In [ ]:
import pandas as pd
import numpy as np

# Pega base com dados pessoais por ID


# Criando DataFrames
A = participants
B = contratos[['id','name','job_title','level','area','department']]

# Realizando o left join 
base_avaliacao_1 = pd.merge(A, B, left_on='reviewer_id', right_on='id', how='left')
base_avaliacao_2 = pd.merge(base_avaliacao_1, B, left_on='reviewee_id', right_on='id', how='left')


base_avals_ind = base_avaliacao_2[['completed','is_complete','finished','survey_participation_id','state','kind',
                                   'anonymized?','relationship',
                                   'id_x','name_x','job_title_x','level_x','area_x','department_x',
                                   'id_y','name_y','job_title_y','level_y','area_y','department_y']]
                                   
                                   
# Dicionário para mapear nomes antigos de colunas para novos nomes
rename_dict = {
    "id_x": "id_avaliador",
    "name_x": "nome_avaliador",
    "job_title_x": "cargo_avaliador",
    "level_x": "nivel_avaliador",
    "area_x": "area_avaliador",
    "department_x": "departamento_avaliador",
    "id_y": "id_avaliado",
    "name_y": "nome_avaliado",
    "job_title_y": "cargo_avaliado",
    "level_y": "nivel_avaliado",
    "area_y": "area_avaliado",
    "department_y": "departamento_avaliado"
}
                                   
 
# Renomeando as colunas do DataFrame
base_avals_ind = base_avals_ind.rename(columns=rename_dict)

                                   
# Salvando o DataFrame em um arquivo Excel
excel_filename = "API_QR_NOTAS/base_avaliacoes_individuais.xlsx"
base_avals_ind.to_excel(excel_filename, index=False)

# Exibindo o resultado
base_avals_ind